# Membrana — wyniki i wydajność MPI

Metoda Jacobiego dla równania $\nabla^2 h=-p/T$, z zerowym brzegiem.
$n$ oznacza liczbę przedziałów na boku, więc wnętrze ma $(n-1)^2$ punktów.
Wczytujemy rzeczywiste logi jednego zadania Slurm. Czas obejmuje obliczenia i komunikację MPI.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

katalog = Path("wyniki/ID_ZADANIA")  # Wpisz katalog pobranego eksperymentu.
dane = pd.read_csv(katalog / "pomiary.csv")
display(dane)


## Mały przykład i weryfikacja

Porównujemy siatki z jednego i wielu procesów. Sprawdzamy zerowy brzeg,
nieujemność ugięcia, symetrię i residuum dyskretnego równania Poissona.
Residuum bliskie zeru potwierdza spełnienie równania na siatce;
próg zmiany iteracji nie jest bezpośrednio błędem rozwiązania ciągłego.


In [ ]:
male = dane[dane["grid_file"].notna()].sort_values("processes")
if male.empty or not (male["processes"] == 1).any() or not (male["processes"] > 1).any():
    raise ValueError("Potrzebne siatki małego przykładu dla P=1 i P>1.")
if not male["converged"].all():
    raise ValueError("Mały przykład nie osiągnął zbieżności.")
if len(male[["n", "a", "p", "tension", "eps"]].drop_duplicates()) != 1:
    raise ValueError("Porównywane siatki muszą mieć jednakowe parametry.")

with np.load(katalog / male.iloc[0]["grid_file"]) as plik:
    wzorzec = plik["h"]

weryfikacja = []
for _, row in male.iterrows():
    with np.load(katalog / row["grid_file"]) as plik:
        h = plik["h"]
    dx = row["a"] / row["n"]
    residuum = (
        h[:-2, 1:-1] + h[2:, 1:-1] + h[1:-1, :-2]
        + h[1:-1, 2:] - 4 * h[1:-1, 1:-1]
    ) / dx**2 + row["p"] / row["tension"]
    brzeg = np.concatenate([h[0], h[-1], h[:, 0], h[:, -1]])
    weryfikacja.append({
        "P": row["processes"],
        "max |brzeg|": np.max(np.abs(brzeg)),
        "min h": h.min(),
        "max h": h.max(),
        "max |h - h(P=1)|": np.max(np.abs(h - wzorzec)),
        "błąd symetrii": max(np.max(np.abs(h - h[::-1])),
                              np.max(np.abs(h - h[:, ::-1]))),
        "max |residuum|": np.max(np.abs(residuum)),
    })
display(pd.DataFrame(weryfikacja))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
obraz = ax.imshow(h, origin="lower", extent=[0, row["a"], 0, row["a"]], cmap="viridis")
ax.set(xlabel="x", ylabel="y", title=f"Ugięcie membrany, n={int(row['n'])}, P={int(row['processes'])}")
fig.colorbar(obraz, ax=ax, label="Ugięcie h(x, y)")
fig.text(0.5, 0.01, "Rys. 1. Ugięcie przy równomiernym obciążeniu i nieruchomym brzegu.",
         ha="center", fontsize=9)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()


## Metryki wydajności

Pomiary ze zapisanymi siatkami służą do weryfikacji i są pomijane w tej analizie.
Dla każdej pary $(n,P)$ przyjmujemy medianę czasów trzech powtórzeń.
Porównujemy uruchomienia o tych samych parametrach fizycznych i progu zbieżności.

$$S(P)=T(1)/T(P),\qquad E(P)=S(P)/P,$$
$$f(P)=\frac{1/S(P)-1/P}{1-1/P}.$$

Dla $P=1$ metryka Karpa–Flatta jest niezdefiniowana.


In [ ]:
pomiary = dane[dane["grid_file"].isna()].copy()
if pomiary.empty:
    raise ValueError("Brak pomiarów wydajności.")
if not pomiary["converged"].all():
    raise ValueError("Nie wszystkie pomiary osiągnęły zbieżność. Powtórz nieudane uruchomienia.")
if len(pomiary[["a", "p", "tension", "eps", "max_iter"]].drop_duplicates()) != 1:
    raise ValueError("Wybierz eksperyment z jednakowymi parametrami.")
if pomiary["n"].nunique() < 2:
    raise ValueError("Potrzebne są co najmniej dwa rozmiary problemu.")
if (pomiary["seconds"] <= 0).any():
    raise ValueError("Czasy muszą być dodatnie.")
if (pomiary.groupby("n")["iterations"].nunique() != 1).any():
    raise ValueError("Dla tego samego n liczba iteracji powinna być jednakowa.")

metryki = pomiary.groupby(["n", "processes"], as_index=False).agg(
    seconds=("seconds", "median"),
    powtorzenia=("seconds", "size"),
    iterations=("iterations", "first"),
)
t1 = metryki[metryki["processes"] == 1].set_index("n")["seconds"]
if not set(metryki["n"]).issubset(t1.index):
    raise ValueError("Brakuje T(1) dla któregoś rozmiaru siatki.")
if (metryki.groupby("n")["processes"].nunique() < 2).any():
    raise ValueError("Każdy rozmiar wymaga pomiarów dla P=1 i P>1.")
metryki["S"] = metryki["n"].map(t1) / metryki["seconds"]
metryki["E"] = metryki["S"] / metryki["processes"]
metryki["f"] = np.nan
wiele = metryki["processes"] > 1
P = metryki.loc[wiele, "processes"]
metryki.loc[wiele, "f"] = (1 / metryki.loc[wiele, "S"] - 1 / P) / (1 - 1 / P)
display(metryki)


In [ ]:
for numer, (kolumna, tytul, os_y) in enumerate([
    ("S", "Przyspieszenie", "S(P) = T(1) / T(P)"),
    ("E", "Efektywność", "E(P) = S(P) / P"),
    ("f", "Metryka Karpa–Flatta", "f(P)"),
], start=2):
    fig, ax = plt.subplots(figsize=(7, 4))
    for n, grupa in metryki.groupby("n"):
        grupa = grupa.sort_values("processes")
        if kolumna == "f":
            grupa = grupa[grupa["processes"] > 1]
        ax.plot(grupa["processes"], grupa[kolumna], "o-", label=f"n={n}, wnętrze {(n-1)**2} punktów")
    ax.set(xlabel="Liczba procesów P", ylabel=os_y, title=tytul)
    ax.set_xticks(sorted(metryki["processes"].unique()))
    ax.grid(alpha=0.3)
    ax.legend()
    fig.text(0.5, 0.01, f"Rys. {numer}. {tytul} dla stałego rozmiaru każdej siatki; mediany czasów.",
             ha="center", fontsize=9)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()


## Omówienie pomiarów

Poniżej podsumowanie najlepszych zmierzonych przyspieszeń dla każdego rozmiaru.
Wzrost $f(P)$ i spadek $E(P)$ wskazują na rosnący udział narzutów,
m.in. wymiany halo i globalnej redukcji w każdej iteracji. Same czasy całkowite
nie pozwalają rozdzielić tych kosztów. Ujemne $f(P)$ oznacza przyspieszenie
ponadliniowe lub zmienność pomiaru — wartości nie są obcinane.


In [ ]:
for n, grupa in metryki.groupby("n"):
    najlepszy = grupa.loc[grupa["S"].idxmax()]
    ostatni = grupa.loc[grupa["processes"].idxmax()]
    print(f"n={n}: najlepsze S={najlepszy['S']:.3f} dla P={int(najlepszy['processes'])}, "
          f"E={najlepszy['E']:.3f}.")
    print(f"  Przy największym P={int(ostatni['processes'])}: "
          f"S={ostatni['S']:.3f}, E={ostatni['E']:.3f}, f={ostatni['f']:.4f}.")
